## Importing necessary libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

## Import Dataset

In [6]:
df=pd.read_csv('./drive/MyDrive/data/fashion.csv')

# EDA

In [4]:
df.head()

,ProductId,Gender,Category,SubCategory,ProductType,Colour,Usage,ProductTitle,Image,ImageURL
0,42419,Girls,Apparel,Topwear,Tops,White,Casual,Gini and Jony Girls Knit White Top,42419.jpg,http://assets.myntassets.com/v1/images/style/p...
1,34009,Girls,Apparel,Topwear,Tops,Black,Casual,Gini and Jony Girls Black Top,34009.jpg,http://assets.myntassets.com/v1/images/style/p...
2,40143,Girls,Apparel,Topwear,Tops,Blue,Casual,Gini and Jony Girls Pretty Blossom Blue Top,40143.jpg,http://assets.myntassets.com/v1/images/style/p...
3,23623,Girls,Apparel,Topwear,Tops,Pink,Casual,Doodle Kids Girls Pink I love Shopping Top,23623.jpg,http://assets.myntassets.com/v1/images/style/p...
4,47154,Girls,Apparel,Bottomwear,Capris,Black,Casual,Gini and Jony Girls Black Capris,47154.jpg,http://assets.myntassets.com/v1/images/style/p...


In [7]:
df.info()
df['Colour'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2906 entries, 0 to 2905
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ProductId     2906 non-null   int64 
 1   Gender        2906 non-null   object
 2   Category      2906 non-null   object
 3   SubCategory   2906 non-null   object
 4   ProductType   2906 non-null   object
 5   Colour        2906 non-null   object
 6   Usage         2906 non-null   object
 7   ProductTitle  2906 non-null   object
 8   Image         2906 non-null   object
 9   ImageURL      2906 non-null   object
dtypes: int64(1), object(9)
memory usage: 227.2+ KB


,count
Colour,
Black,578
White,513
Blue,338
Brown,220
Red,185
Pink,184
Green,143
Grey,127
Yellow,127


The `TripletDataset` expects a specific directory structure with `anchor`, `positive`, and `negative` subdirectories. The previous error indicated that these directories were not found. I will create the necessary empty directory structure here.

In [6]:
import os

def create_triplet_dirs(base_path):
    """Creates the required anchor, positive, and negative subdirectories."""
    for sub_dir_name in ['anchor', 'positive', 'negative']:
        path = os.path.join(base_path, sub_dir_name)
        os.makedirs(path, exist_ok=True)
        print(f"Created directory: {path}")

# Define the base paths for training and testing data
train_base_path = './drive/MyDrive/data/Apparel/Boys/'
test_base_path = './drive/MyDrive/data/Apparel/Boys/'

print("Creating directories for training data...")
create_triplet_dirs(train_base_path)

print("\nCreating directories for testing data...")
create_triplet_dirs(test_base_path)

print("\nDirectory structure created.")

Creating directories for training data...
Created directory: ./drive/MyDrive/data/Apparel/Boys/anchor
Created directory: ./drive/MyDrive/data/Apparel/Boys/positive
Created directory: ./drive/MyDrive/data/Apparel/Boys/negative

Creating directories for testing data...
Created directory: ./drive/MyDrive/data/Apparel/Boys/anchor
Created directory: ./drive/MyDrive/data/Apparel/Boys/positive
Created directory: ./drive/MyDrive/data/Apparel/Boys/negative

Directory structure created.


### Generating Triplet Dataset Structure

To ensure the `TripletDataset` can find the necessary images, we need to populate the `anchor`, `positive`, and `negative` subdirectories. The `generate_triplets` function below will do this by:

1.  **Defining Similarity**: For an anchor image, a 'positive' image will share the same `Category`, `SubCategory`, and `Colour` (based on your criteria).
2.  **Defining Dissimilarity**: A 'negative' image will have at least one different attribute (Category, SubCategory, or Colour) compared to the anchor.
3.  **Copying Images**: It will copy the relevant images from a specified `original_image_folder` into the `anchor`, `positive`, and `negative` directories.

In [14]:
import shutil
import random

def generate_triplets(df, original_image_folder, target_base_dir):
    """
    Generates triplet (anchor, positive, negative) image paths based on similarity criteria
    and copies images to target subdirectories.

    Args:
        df (pd.DataFrame): DataFrame containing product metadata.
        original_image_folder (str): Path to the directory containing all original images.
        target_base_dir (str): Base directory where 'anchor', 'positive', 'negative'
                                subdirectories will be created/populated.
    """
    print(f"Generating triplets for: {target_base_dir}")

    anchor_dir = os.path.join(target_base_dir, 'anchor')
    positive_dir = os.path.join(target_base_dir, 'positive')
    negative_dir = os.path.join(target_base_dir, 'negative')

    # Ensure target directories exist and are empty
    for d in [anchor_dir, positive_dir, negative_dir]:
        if os.path.exists(d):
            shutil.rmtree(d) # Clear existing content
        os.makedirs(d, exist_ok=True)

    # Convert ProductId to string for image filenames
    df['ImageFileName'] = df['ProductId'].astype(str) + '.jpg'

    # Filter out products where image file does not exist in original_image_folder
    if not os.path.exists(original_image_folder):
        print(f"Error: Original image folder '{original_image_folder}' does not exist. Please create it and place your images there.")
        return

    available_images = os.listdir(original_image_folder)
    df_filtered = df[df['ImageFileName'].isin(available_images)].copy()

    if df_filtered.empty:
        print(f"No product images found in '{original_image_folder}' matching DataFrame entries. Please check path and filenames.")
        return

    triplet_count = 0
    skipped_anchors = 0

    # Group DataFrame for efficient positive/negative sampling
    # For positive samples, we need items with the exact same attributes
    # For negative samples, we need items with different attributes

    for idx, anchor_row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc=f"Processing images for {target_base_dir}"):
        anchor_product_id = anchor_row['ProductId']
        anchor_image_name = anchor_row['ImageFileName']
        anchor_category = anchor_row['Category']
        anchor_subcategory = anchor_row['SubCategory']
        anchor_colour = anchor_row['Colour']

        anchor_src_path = os.path.join(original_image_folder, anchor_image_name)

        # Find a positive example: same category, subcategory, colour, different product ID
        positive_candidates = df_filtered[
            (df_filtered['Category'] == anchor_category) &
            (df_filtered['SubCategory'] == anchor_subcategory) &
            (df_filtered['Colour'] == anchor_colour) &
            (df_filtered['ProductId'] != anchor_product_id)
        ]

        if positive_candidates.empty:
            skipped_anchors += 1
            continue # Cannot find a suitable positive match

        positive_row = positive_candidates.sample(1).iloc[0]
        positive_image_name = positive_row['ImageFileName']
        positive_src_path = os.path.join(original_image_folder, positive_image_name)

        # Find a negative example: try to maximize dissimilarity
        # Prioritize different category, then subcategory, then colour
        negative_candidates = df_filtered[
            (df_filtered['Category'] != anchor_category) # Different category
        ]
        if negative_candidates.empty:
            negative_candidates = df_filtered[
                (df_filtered['SubCategory'] != anchor_subcategory) & # Different subcategory, same category
                (df_filtered['Category'] == anchor_category)
            ]
        if negative_candidates.empty:
            negative_candidates = df_filtered[
                (df_filtered['Colour'] != anchor_colour) & # Different colour, same category/subcategory
                (df_filtered['Category'] == anchor_category) &
                (df_filtered['SubCategory'] == anchor_subcategory)
            ]
        if negative_candidates.empty:
            # Fallback: any product that is not the anchor itself (should rarely happen with enough data)
            negative_candidates = df_filtered[
                (df_filtered['ProductId'] != anchor_product_id)
            ]

        if negative_candidates.empty:
            skipped_anchors += 1
            continue # Cannot find a suitable negative match

        negative_row = negative_candidates.sample(1).iloc[0]
        negative_image_name = negative_row['ImageFileName']
        negative_src_path = os.path.join(original_image_folder, negative_image_name)

        # Copy images to target directories, ensuring positive and negative are also named after the anchor.
        # This is crucial for TripletDataset to find corresponding files.
        shutil.copy(anchor_src_path, os.path.join(anchor_dir, anchor_image_name))
        shutil.copy(positive_src_path, os.path.join(positive_dir, anchor_image_name)) # Positive image also named as anchor
        shutil.copy(negative_src_path, os.path.join(negative_dir, anchor_image_name)) # Negative image also named as anchor
        triplet_count += 1

    print(f"Generated {triplet_count} triplets in {target_base_dir}.")
    if skipped_anchors > 0:
        print(f"Skipped {skipped_anchors} anchors due to inability to find suitable positive or negative matches.")

# --- Call the triplet generation function ---
# Define the path where all original images are located.
# IMPORTANT: Please ensure your image files (e.g., '12345.jpg') are in this directory.
original_image_folder = './drive/MyDrive/data/Apparel/Boys/' # Assuming this path for original images

# The base paths for storing generated triplets.
# These were defined in a previous cell.
train_base_path = './drive/MyDrive/data/Apparel/Boys/'
# test_base_path = './drive/MyDrive/data/Apparel/Boys/'

# Call the function to generate triplets for the training base path
generate_triplets(df, original_image_folder, train_base_path)

# If you have separate data for validation/testing, you would call it again for test_base_path
# For now, we'll assume train_base_path is sufficient for demonstration/initial setup.
# If your test data is distinct, you'd need a separate df_test and original_image_folder_test
# generate_triplets(df_test, original_image_folder_test, test_base_path)

Generating triplets for: ./drive/MyDrive/data/Apparel/Boys/
No product images found in './drive/MyDrive/data/Apparel/Boys/' matching DataFrame entries. Please check path and filenames.


In [10]:
import shutil
import os
# This function is being replaced by the more robust `generate_triplets` function above.
# The logic here has several issues, including the 'ValueError: The truth value of a Series is ambiguous.'
# and incorrect triplet generation logic.
# For proper triplet generation, please refer to the `generate_triplets` function.
def generateAPN_deprecated(categoryPath:str): ##used to automate separation of anchor,positive and negative
  for i in range(len(df)): ## since all images in the anchor have the same Category and gender,we compare just color and SubCategory
    ## imageName:str=str(df.iloc[i]['ProductId'])+'.jpg'
    for filename in os.listdir(categoryPath):
      ProID=int(filename.replace(".jpg",""))
      # Access the row using .loc or .iloc to avoid Series ambiguity
      product_row = df[df['ProductId']==ProID].iloc[0]

      # Incorrect logic for positive: should be copy, not move, and needs an anchor
      if product_row['Category']==df.iloc[i]['Category'] and product_row['Colour']==df.iloc[i]['Colour'] and product_row['SubCategory']==df.iloc[i]['SubCategory']:
        # This logic is problematic for triplet generation. It moves the file if it matches
        # the criteria for a positive, but doesn't handle the anchor and negative parts.
        pass
        # shutil.move(categoryPath+'/'+filename,train_base_path+'/positive'+'/'+filename)

      # Incorrect logic for negative: should be 'AND NOT', not 'OR'
      if product_row['Category']==df.iloc[i]['Category'] or product_row['Colour']==df.iloc[i]['Colour'] or product_row['SubCategory']==df.iloc[i]['SubCategory'] :
        # This condition will almost always be true, making most items 'negative' incorrectly.
        # A negative example should be dissimilar to the anchor.
        pass
        # shutil.move(categoryPath+'/'+filename,train_base_path+'/negative'+'/'+filename)


In [ ]:
# The generateAPN function is deprecated and replaced by generate_triplets.
# Please ensure the generate_triplets function (defined in a previous cell)
# has been executed to populate the triplet directories.
# generateAPN("/content/drive/MyDrive/data/Apparel/Boys/anchor")


In [ ]:
class L2Normalize(nn.Module):
    def __init__(self, p=2, dim=1, eps=1e-12):
        super().__init__()
        self.p = p
        self.dim = dim
        self.eps = eps

    def forward(self, x):
        return nn.functional.normalize(x, p=self.p, dim=self.dim, eps=self.eps)

In [ ]:
def get_resnet50_encoder(embedding_dim=512, pretrained=True, train_backbone=False):
    resnet = models.resnet50(pretrained=pretrained)
    modules = list(resnet.children())[:-1]  # Remove last FC layer
    backbone = nn.Sequential(*modules)

    model = nn.Sequential(
        backbone,
        nn.Flatten(),
        nn.Linear(2048, embedding_dim),
        L2Normalize()  # L2 normalize embeddings
    )

    # Optionally freeze backbone
    if not train_backbone:
        for param in backbone.parameters():
            param.requires_grad = False

    return model

In [ ]:
# ===== 2. Dataset & DataLoader =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import os
import shutil

# The rename_triplet_files function is no longer needed.
# The `generate_triplets` function already handles naming the positive and negative
# images identically to their anchor counterpart when they are copied into the
# respective directories, which is the required format for TripletDataset.
# def rename_triplet_files(base_dir):
#     """
#     Renames files in 'positive' and 'negative' subdirectories to match
#     corresponding files in the 'anchor' subdirectory, assuming a positional match.

#     Args:
#         base_dir (str): The root directory containing 'anchor', 'positive',
#                         and 'negative' subdirectories (e.g., './train/Footwear/Men/').
#     """
#     anchor_dir = os.path.join(base_dir, 'anchor')
#     positive_dir = os.path.join(base_dir, 'positive')
#     negative_dir = os.path.join(base_dir, 'negative')

#     if not all(os.path.isdir(d) for d in [anchor_dir, positive_dir, negative_dir]):
#         print(f"Error: One or more triplet directories not found in {base_dir}")
#         return

#     # Get sorted lists of files from each directory
#     anchor_files = sorted([f for f in os.listdir(anchor_dir) if os.path.isfile(os.path.join(anchor_dir, f))])
#     positive_files = sorted([f for f in os.listdir(positive_dir) if os.path.isfile(os.path.join(positive_dir, f))])
#     negative_files = sorted([f for f in os.listdir(negative_dir) if os.path.isfile(os.path.join(negative_dir, f))])

#     if not (len(anchor_files) == len(positive_files) == len(negative_files)):
#         print("\nWARNING: Number of files in anchor, positive, and negative directories do not match.")
#         print(f"Anchor files: {len(anchor_files)}, Positive files: {len(positive_files)}, Negative files: {len(negative_files)}")
#         print("Proceeding with the minimum count, but this might lead to incorrect pairings or leave some files unrenamed.")
#         min_count = min(len(anchor_files), len(positive_files), len(negative_files))
#     else:
#         min_count = len(anchor_files)

#     print(f"\nAttempting to rename {min_count} triplets in '{base_dir}'...")
#     renamed_count = 0
#     for i in range(min_count):
#         anchor_name = anchor_files[i]
#         old_positive_name = positive_files[i]
#         old_negative_name = negative_files[i]

#         # Define new paths for positive and negative files
#         new_positive_path = os.path.join(positive_dir, anchor_name)
#         new_negative_path = os.path.join(negative_dir, anchor_name)

#         # Define current paths for positive and negative files
#         old_positive_path = os.path.join(positive_dir, old_positive_name)
#         old_negative_path = os.path.join(negative_dir, old_negative_name)

#         # Rename positive file
#         if old_positive_name != anchor_name: # Only rename if name is different
#             if os.path.exists(new_positive_path):
#                 print(f"  Skipping positive rename for '{anchor_name}': Target path '{new_positive_path}' already exists. Move/delete it first if you want to replace.")
#             else:
#                 try:
#                     os.rename(old_positive_path, new_positive_path)
#                     print(f"  Renamed positive: '{old_positive_name}' to '{anchor_name}'")
#                     renamed_count += 1
#                 except OSError as e:
#                     print(f"  Error renaming positive file '{old_positive_name}' to '{anchor_name}': {e}")

#         # Rename negative file
#         if old_negative_name != anchor_name: # Only rename if name is different
#             if os.path.exists(new_negative_path):
#                 print(f"  Skipping negative rename for '{anchor_name}': Target path '{new_negative_path}' already exists. Move/delete it first if you want to replace.")
#             else:
#                 try:
#                     os.rename(old_negative_path, new_negative_path)
#                     print(f"  Renamed negative: '{old_negative_name}' to '{anchor_name}'")
#                     renamed_count += 1
#                 except OSError as e:
#                     print(f"  Error renaming negative file '{old_negative_name}' to '{anchor_name}': {e}")

#     print(f"\nFinished renaming process for '{base_dir}'. Total files renamed: {renamed_count}")

# Example Usage: (These calls are now commented out as the function is deprecated)
# print("Renaming files for training data...")
# rename_triplet_files("./train/Footwear/Men/")

# print("\nRenaming files for validation data...")
# rename_triplet_files("./test/Footwear/Men/")


Renaming files for training data...

Attempting to rename 0 triplets in './train/Footwear/Men/'...

Finished renaming process for './train/Footwear/Men/'. Total files renamed: 0

Renaming files for validation data...

Attempting to rename 0 triplets in './test/Footwear/Men/'...

Finished renaming process for './test/Footwear/Men/'. Total files renamed: 0


In [ ]:
import os
from PIL import Image
import random
from torch.utils.data import Dataset
from torchvision import transforms

class TripletDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.anchor_dir = os.path.join(root_dir, 'anchor')
        self.positive_dir = os.path.join(root_dir, 'positive')
        self.negative_dir = os.path.join(root_dir, 'negative')

        if not os.path.isdir(self.anchor_dir) or \
           not os.path.isdir(self.positive_dir) or \
           not os.path.isdir(self.negative_dir):
            raise RuntimeError(f"Expected 'anchor', 'positive', and 'negative' subdirectories in {root_dir}")

        self.triplets = self._find_triplets()

        if not self.triplets:
            raise RuntimeError(f"Found 0 valid triplets in {root_dir}. Ensure corresponding images exist in anchor, positive, and negative subdirectories.")

    def _find_triplets(self):
        triplets = []
        anchor_images = sorted([f for f in os.listdir(self.anchor_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])

        for img_name in anchor_images:
            positive_path = os.path.join(self.positive_dir, img_name)
            negative_path = os.path.join(self.negative_dir, img_name)

            if os.path.exists(positive_path) and os.path.exists(negative_path):
                triplets.append((os.path.join(self.anchor_dir, img_name), positive_path, negative_path))
        return triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor_path, positive_path, negative_path = self.triplets[idx]

        anchor_img = Image.open(anchor_path).convert("RGB")
        positive_img = Image.open(positive_path).convert("RGB")
        negative_img = Image.open(negative_path).convert("RGB")

        if self.transform:
            anchor_img = self.transform(anchor_img)
            positive_img = self.transform(positive_img)
            negative_img = self.transform(negative_img)

        return anchor_img, positive_img, negative_img

In [ ]:
# Update the dataset and dataloader creation with the new TripletDataset

try:
    # Use the defined train_base_path and test_base_path for TripletDataset
    train_dataset = TripletDataset(train_base_path, transform=transform)
    val_dataset = TripletDataset(test_base_path, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    print("Triplet Datasets and DataLoaders created successfully!")
except RuntimeError as e:
    print(f"Error creating Triplet Dataset: {e}")
    print("Please ensure your root directories (e.g., ./drive/MyDrive/data/Apparel/Boys/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.")
    print("Example: ./drive/MyDrive/data/Apparel/Boys/anchor/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/positive/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/negative/image1.jpg")

Error creating Triplet Dataset: Found 0 valid triplets in ./train/Footwear/Men/. Ensure corresponding images exist in anchor, positive, and negative subdirectories.
Please ensure your root directories (e.g., ./train/Footwear/Men/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.
Example: ./train/Footwear/Men/anchor/image1.jpg, ./train/Footwear/Men/positive/image1.jpg, ./train/Footwear/Men/negative/image1.jpg


In [ ]:
import os
import shutil

# Path to the .ipynb_checkpoints directory that might be causing issues
checkpoint_dir_train = os.path.join(train_base_path, '.ipynb_checkpoints')
checkpoint_dir_val = os.path.join(test_base_path, '.ipynb_checkpoints')

# Check if the directory exists and remove it
if os.path.exists(checkpoint_dir_train) and os.path.isdir(checkpoint_dir_train):
    shutil.rmtree(checkpoint_dir_train)
    print(f"Removed problematic directory: {checkpoint_dir_train}")

if os.path.exists(checkpoint_dir_val) and os.path.isdir(checkpoint_dir_val):
    shutil.rmtree(checkpoint_dir_val)
    print(f"Removed problematic directory: {checkpoint_dir_val}")

try:
    # Use the defined train_base_path and test_base_path for TripletDataset
    train_dataset = TripletDataset(train_base_path, transform=transform)
    val_dataset = TripletDataset(test_base_path, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    print("Triplet Datasets and DataLoaders created successfully!")
except RuntimeError as e:
    print(f"Error creating Triplet Dataset: {e}")
    print("Please ensure your root directories (e.g., ./drive/MyDrive/data/Apparel/Boys/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.")
    print("Example: ./drive/MyDrive/data/Apparel/Boys/anchor/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/positive/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/negative/image1.jpg")

# ===== 3. Model, Loss, Optimizer =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_resnet50_encoder(embedding_dim=256, pretrained=True, train_backbone=True).to(device)

# Example: contrastive learning loss (InfoNCE, Triplet, etc.)
# Here we use TripletMarginLoss for demonstration
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

Error creating Triplet Dataset: Found 0 valid triplets in ./train/Footwear/Men/. Ensure corresponding images exist in anchor, positive, and negative subdirectories.
Please ensure your root directories (e.g., ./train/Footwear/Men/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.
Example: ./train/Footwear/Men/anchor/image1.jpg, ./train/Footwear/Men/positive/image1.jpg, ./train/Footwear/Men/negative/image1.jpg


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        # For Triplet loss, you need (anchor, positive, negative) samples
        # Here we assume you have a custom dataset that returns them
        # This is just a placeholder
        anchor, positive, negative = batch  # Replace with your triplet dataset
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        optimizer.zero_grad()
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            anchor, positive, negative = batch
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)

            loss = criterion(emb_a, emb_p, emb_n)
            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = validate_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

# ===== 7. Save the trained encoder =====
torch.save(model.state_dict(), "resnet50_encoder.pth")
print("Model saved as resnet50_encoder.pth")

NameError: name 'train_loader' is not defined

In [ ]:
import matplotlib.pyplot as plt

# Plotting the training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss')
plt.plot(range(1, EPOCHS + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()
